# Этап 1. Скрининг рычагов L2–L6

**Вопрос этапа:** какие рычаги и какие их взаимодействия существенно влияют на K0–K5, и при каком шоке. Ответ определяет, какие рычаги идут на этап 2 (подбор сил) и какие комбинации можно отбросить сразу.

**План (зафиксирован до прогонов):**
* полный факторный план 2⁵ по переключателям L2, L3, L4, L5, L6; уровни −1 = выключен, +1 = включён с силой из модели (L3-share 0.15, L4-limit 0.10, L5 30 L, L6 1.1);
* три сценария `shock-depth` 0.2 / 0.45 / 0.7 - анализ по каждому отдельно;
* R = 5 повторов, первые 5 seed пула `screening`, одни и те же для всех ячеек (общие случайные числа); seed - блок в модели анализа;
* порядок прогонов по seed: при остановке раньше срока остаётся полный сбалансированный план с меньшим R;
* +5 прогонов без шока и без мер (фон очередей K5);
* итого 32 × 3 × 5 + 5 = **485 прогонов**.

**Анализ:** регрессия `log(1 + K)` на кодированные факторы со всеми взаимодействиями до 3-го порядка + блок seed; ANOVA с частичной η² как мерой важности; диаграммы Парето эффектов; графики взаимодействий; тепловая карта важности; таблица 32 комбинаций и недоминируемые комбинации.

**Правило отбора (зафиксировано до прогонов):** рычаг идёт на этап 2, если хотя бы один член модели с его участием имеет η² ≥ 0.05 хотя бы для одного KPI хотя бы при одном уровне шока. Комбинация отбрасывается, если она доминируема при всех трёх уровнях шока.

Ноутбук лежит в той же папке, что и ноутбук этапа 0 (нужны `abm_runner.py` и папка `results`).

## 1.0. Окружение и паспорт этапа 0

In [1]:
# %pip install statsmodels psutil   # один раз, если не установлены

import json, hashlib, itertools, math, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import abm_runner as ar

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.dpi": 110, "axes.prop_cycle": plt.cycler(color=PAL)})


In [2]:
NETLOGO_HOME = r"C:\Program Files\NetLogo 7.0.4"
MODEL_PATH   = Path("refinery_shock_abm.nlogox")
OUT_DIR      = Path("results")


# preparation results from stage 0
PREP  = json.loads((OUT_DIR / "prep_summary.json").read_text(encoding="utf-8"))

# Load the seeds generated in stage 0
SEEDS = json.loads((OUT_DIR / "seeds.json").read_text(encoding="utf-8"))["pools"]

# check if the model file has not changed since stage 0
sha = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
assert sha == PREP["model_sha256"], "File changed"
print("The same model as in stage 0:", sha[:16], "| NetLogo", PREP["netlogo_version"])

The same model as in stage 0: 2b246166dc0c1182 | NetLogo 7.0.4


In [3]:
R          = 5  # replications per shock level
SCR_SEEDS  = SEEDS["screening"][:R]
SHOCKS     = PREP["shock_levels"]                     
LEVER_ON   = PREP["lever_on"]
SWITCH     = {"L2": "L2-export-ban?", "L3": "L3-quality-relax?", "L4": "L4-import?",
              "L5": "L5-fillup-limit?", "L6": "L6-price-cap?"}
LEVERS     = list(SWITCH)
KPIS       = ["K0", "K0@90", "K1", "K2", "K3_volume", "K4", "K5_hours"]
N_CARS     = PREP["constants"]["n-consumers"]
D0         = PREP["constants"]["D0"]
ETA2_MIN   = 0.05  # the minimum value of the uncertain parameter eta2                                    

print("scrining seed :", SCR_SEEDS)

scrining seed : [347, 164, 159, 797, 98]


In [4]:
# JVM arguments for the Java Virtual Machine
# -Xmx512m sets the maximum heap size to 512 megabytes
JVM_ARGS = ["-Xmx512m"]    
CFG = {"netlogo_home": NETLOGO_HOME, "model_path": str(MODEL_PATH.resolve()), "jvmargs": JVM_ARGS,
       "jvm_path": ar.find_jvm(NETLOGO_HOME), "horizon": PREP["horizon"], "checkpoints": PREP["checkpoints"],
       "interface_defaults": PREP["interface_defaults"], "nominal_uncertain": PREP["nominal_uncertain"]}


## 1.1. Таблица плана

32 комбинации называются по включённым рычагам (`none` - все выключены). Столбцы `L2`…`L6` - кодированные уровни −1/+1, по ним идёт анализ. В заданиях явно выставлены все переключатели и силы, поэтому строки CSV одинаковы по составу столбцов.

In [5]:
def combo_name(bits):
    on = [l for l, b in zip(LEVERS, bits) if b]
    return "+".join(on) if on else "none"

def make_job(bits, shock, seed, rep, stage="screening"):
    params = {SWITCH[l]: bool(b) for l, b in zip(LEVERS, bits)} | LEVER_ON | {
        "L4-subsidy": PREP["interface_defaults"]["L4-subsidy"], "shock-depth": shock}
    name = combo_name(bits)
    return {"run_id": f"{stage}|{name}|sd={shock}|seed={seed}", "stage": stage, "design_id": name,
            "code": "".join(map(str, bits)), "shock": shock, "seed": int(seed), "rep": rep,
            **{l: (1 if b else -1) for l, b in zip(LEVERS, bits)}, "params": params, "uncertain": None}

COMBOS = list(itertools.product([0, 1], repeat=5))
JOBS = [make_job((0,) * 5, 0.0, s, r, stage="background") for r, s in enumerate(SCR_SEEDS, 1)]
for r, s in enumerate(SCR_SEEDS, 1):                 # по seed: полный план для каждого повтора
    for sh in SHOCKS:
        for bits in COMBOS:
            JOBS.append(make_job(bits, sh, s, r))

DESIGN = pd.DataFrame([{k: v for k, v in j.items() if k not in ("params", "uncertain")} for j in JOBS])
DESIGN.to_csv(OUT_DIR / "design_screening.csv", index=False, encoding="utf-8-sig")
assert DESIGN.run_id.is_unique
X = DESIGN.query("stage == 'screening' and rep == 1 and shock == @SHOCKS[0]")[LEVERS].to_numpy()
assert np.allclose(X.T @ X, 32 * np.eye(5)), "план не ортогонален"
print(f"заданий: {len(JOBS)} (план {len(JOBS) - R} + фон {R})")
DESIGN.head(8)

заданий: 485 (план 480 + фон 5)


,run_id,stage,design_id,code,shock,seed,rep,L2,L3,L4,L5,L6
0,background|none|sd=0.0|seed=347,background,none,00000,0.0,347,1,-1,-1,-1,-1,-1
1,background|none|sd=0.0|seed=164,background,none,00000,0.0,164,2,-1,-1,-1,-1,-1
2,background|none|sd=0.0|seed=159,background,none,00000,0.0,159,3,-1,-1,-1,-1,-1
3,background|none|sd=0.0|seed=797,background,none,00000,0.0,797,4,-1,-1,-1,-1,-1
4,background|none|sd=0.0|seed=98,background,none,00000,0.0,98,5,-1,-1,-1,-1,-1
5,screening|none|sd=0.2|seed=347,screening,none,00000,0.2,347,1,-1,-1,-1,-1,-1
6,screening|L6|sd=0.2|seed=347,screening,L6,00001,0.2,347,1,-1,-1,-1,-1,1
7,screening|L5|sd=0.2|seed=347,screening,L5,00010,0.2,347,1,-1,-1,-1,1,-1


## 1.2. Сколько процессов запускать

В этапе 0 тест сравнивал пул с одним быстрым прогоном (8.2 с), а средний прогон в процессе ноутбука - 22 с. Пересчёт того же теста: 4 процесса давали ~385 прогонов/ч против ~164 в одиночку, то есть пул всё-таки ускорял в ~2.3 раза.

Здесь тест честный: 8 заданий (4 комбинации × 2 seed пилота), время которых в одиночном режиме уже известно из `pilot.csv`. Для 2, 3, 4 процессов считаем пропускную способность в установившемся режиме (`число процессов × 3600 / среднее время прогона в пуле`) и сравниваем с одиночной на тех же заданиях. Заодно проверяем, что результаты в пуле совпадают с пилотом (воспроизводимость между процессами).

**Перед запуском:** закрыть браузер и тяжёлые программы, ноутбук на зарядке. Java в процессе ноутбука на этом шаге не запускается. Тест займёт ~8-10 мин; ячейку можно пропустить и задать `N_WORKERS` вручную в следующей.

In [6]:
import importlib
import batch_runner as br
importlib.reload(br)

<module 'batch_runner' from 'c:\\Users\\arsmi\\Downloads\\netlogo-project\\Experiments\\batch_runner.py'>

In [7]:
import psutil
vm = psutil.virtual_memory()
print(f"физических ядер: {psutil.cpu_count(logical=False)}, RAM свободно {vm.available / 2**30:.1f} из {vm.total / 2**30:.1f} ГБ")

PILOT = br.read_results(OUT_DIR / "pilot.csv")
pilot_map = {"P0": (0, 0, 0, 0, 0), "P1": (0, 1, 0, 1, 1), "P2": (1, 0, 1, 0, 0), "ALL": (1, 1, 1, 1, 1)}
test_seeds = SEEDS["pilot"][:2]
ref = PILOT[PILOT.seed.isin(test_seeds)].groupby(["design_id", "seed"]).mean(numeric_only=True)

test_rows = []
for n in [2, 3, 4]:
    if psutil.virtual_memory().available / 2**30 < 0.7 * n:
        print(f"{n} процессов: мало свободной памяти, пропускаю"); continue
    tjobs = []
    for pid, bits in pilot_map.items():
        for s in test_seeds:
            j = make_job(bits, PREP["shock_mid"], s, 0, stage="ptest")
            j["run_id"] += f"|w{n}"; j["pilot_id"] = pid
            tjobs.append(j)
    out = OUT_DIR / f"_pool_test_w{n}.csv"
    out.unlink(missing_ok=True)
    t0 = time.perf_counter()
    res = br.run_jobs(tjobs, CFG, out, n_workers=n, log_every=30, resume=False)
    wall = time.perf_counter() - t0
    single = np.mean([ref.loc[(p, s), "sec"] for p, s in zip(res.pilot_id, res.seed)])
    same = all(np.isclose(r.K0, ref.loc[(r.pilot_id, r.seed), "K0"], rtol=1e-9) and
               np.isclose(r.K4, ref.loc[(r.pilot_id, r.seed), "K4"], rtol=1e-9) for r in res.itertuples())
    test_rows.append({"процессов": n, "с/прогон в пуле": res.sec.mean(), "с/прогон один": single,
                      "прогонов/ч (пул)": n * 3600 / res.sec.mean(), "прогонов/ч (один)": 3600 / single,
                      "ускорение": n * single / res.sec.mean(), "мин на тест": wall / 60,
                      "совпадает с пилотом": same})
    out.unlink(missing_ok=True)
POOL_TEST = pd.DataFrame(test_rows)
POOL_TEST.round(2)

физических ядер: 4, RAM свободно 1.9 из 15.8 ГБ
16:29:07 старт: 8 прогонов, процессов: 2


16:29:37 ... работаю, 0/8
16:29:55 1/8 | 1 мин | 76 прогонов/ч | осталось ~6 мин | ошибок 0
16:30:34 5/8 | 1 мин | 208 прогонов/ч | осталось ~1 мин | ошибок 0
16:30:56 8/8 | 2 мин | 263 прогонов/ч | осталось ~0 мин | ошибок 0
16:30:56 готово: 8 за 1.8 мин, ошибок 0
16:30:56 старт: 8 прогонов, процессов: 3
16:31:27 ... работаю, 0/8
16:31:57 ... работаю, 0/8
16:32:03 1/8 | 1 мин | 54 прогонов/ч | осталось ~8 мин | ошибок 0
16:32:44 5/8 | 2 мин | 167 прогонов/ч | осталось ~1 мин | ошибок 0
16:33:03 8/8 | 2 мин | 227 прогонов/ч | осталось ~0 мин | ошибок 0
16:33:03 готово: 8 за 2.1 мин, ошибок 0
4 процессов: мало свободной памяти, пропускаю


,процессов,с/прогон в пуле,с/прогон один,прогонов/ч (пул),прогонов/ч (один),ускорение,мин на тест,совпадает с пилотом
0,2,20.60,29.02,349.48,124.04,2.82,1.82,True
1,3,32.64,29.02,330.93,124.04,2.67,2.11,True


In [8]:
# выбор: лучший вариант, если ускорение хотя бы 1.3; иначе один процесс
if len(POOL_TEST) and POOL_TEST["ускорение"].max() >= 1.3:
    best = POOL_TEST.loc[POOL_TEST["ускорение"].idxmax()]
    N_WORKERS, RUNS_PER_HOUR = int(best["процессов"]), float(best["прогонов/ч (пул)"])
else:
    N_WORKERS, RUNS_PER_HOUR = 1, 3600 / 22
# N_WORKERS, RUNS_PER_HOUR = 3, 380      # <- ручная установка, если тест пропущен
print(f"N_WORKERS = {N_WORKERS}, ~{RUNS_PER_HOUR:.0f} прогонов/ч -> 485 прогонов за ~{485 / RUNS_PER_HOUR:.1f} ч")

N_WORKERS = 2, ~349 прогонов/ч -> 485 прогонов за ~1.4 ч


## 1.3. Запуск

Результаты дописываются в `results/screening.csv` каждые 10 прогонов. Если ячейку прервать (Kernel → Interrupt) или ноутбук перезапустится, просто запустите ячейку снова: посчитанные `run_id` пропускаются. Журнал печатается раз в минуту. Пока идут прогоны, Windows не уйдёт в сон (крышку не закрывать, если в настройках питания закрытие = сон).

In [ ]:
link = ar.make_link(CFG) if N_WORKERS == 1 else None
RES = br.run_jobs(JOBS, CFG, OUT_DIR / "screening.csv", n_workers=N_WORKERS, link=link, log_every=60)
print(RES.groupby("stage").size())

16:33:03 старт: 485 прогонов, процессов: 2
16:34:11 3/485 | 1 мин | 159 прогонов/ч | осталось ~182 мин | ошибок 0
16:35:27 9/485 | 2 мин | 225 прогонов/ч | осталось ~127 мин | ошибок 0
16:36:37 15/485 | 4 мин | 253 прогонов/ч | осталось ~112 мин | ошибок 0
16:37:43 23/485 | 5 мин | 296 прогонов/ч | осталось ~94 мин | ошибок 0
16:38:45 31/485 | 6 мин | 327 прогонов/ч | осталось ~83 мин | ошибок 0
16:39:49 39/485 | 7 мин | 347 прогонов/ч | осталось ~77 мин | ошибок 0
16:41:01 45/485 | 8 мин | 339 прогонов/ч | осталось ~78 мин | ошибок 0
16:42:15 49/485 | 9 мин | 320 прогонов/ч | осталось ~82 мин | ошибок 0
16:43:25 54/485 | 10 мин | 313 прогонов/ч | осталось ~83 мин | ошибок 0
16:44:35 59/485 | 12 мин | 307 прогонов/ч | осталось ~83 мин | ошибок 0
16:45:48 65/485 | 13 мин | 306 прогонов/ч | осталось ~82 мин | ошибок 0
16:47:23 71/485 | 14 мин | 297 прогонов/ч | осталось ~84 мин | ошибок 0
16:48:37 74/485 | 16 мин | 285 прогонов/ч | осталось ~86 мин | ошибок 0
16:49:38 78/485 | 17 мин | 2

## 1.4. Check for completeness and reproducibility

* all 485 tasks completed, no errors
* 15 runs of 'none' match the runs of stage 0 (`baseline_P0.csv`) - the model, seed, and connection to NetLogo yield the same result on a different day and in a different process;
* run time by combinations - will be useful for the budget of stages 2-3.

In [ ]:
RES = br.read_results(OUT_DIR / "screening.csv")
missing = set(DESIGN.run_id) - set(RES.run_id)
print(f"calculated {RES.run_id.isin(DESIGN.run_id).sum()} out of {len(DESIGN)}, missing {len(missing)}")
err_file = OUT_DIR / "screening.errors.csv"
print("error file:", "exists - check it" if err_file.exists() else "does not exist")

# check that the baseline results match the 'none' design in the screening stage
BASE0 = br.read_results(OUT_DIR / "baseline_P0.csv")
chk = RES.query("design_id == 'none' and stage == 'screening'").merge(
    BASE0[["shock", "seed", "K0", "K1", "K4", "K5_hours"]], on=["shock", "seed"], suffixes=("", "_b0"))
ok = all(np.allclose(chk[k], chk[k + "_b0"], rtol=1e-9) for k in ["K0", "K1", "K4", "K5_hours"])
print(f"match 'none' with stage 0: {ok} ({len(chk)} runs)")

# analyze the screening results
SCR = RES[RES.stage == "screening"].copy()
BG = RES[RES.stage == "background"].copy()
print(f"run time: average {SCR.sec.mean():.1f} s, median {SCR.sec.median():.1f} s")
SCR.groupby("design_id").sec.mean().sort_values(ascending=False).head(6).round(1)

## 1.5. Analysis of the screening results

* `K5_excess` - queues above the background without a shock, time on the machine: K5 at the background is almost the same as at 'none' with a weak and moderate shock, so we compare what the shock and levers add;
* `K2` is recalculated for any subsidy: K2 = L4-subsidy × import volume, import volume = K2 / 0.25;
* for analysis we use `log(1 + K)` since the effects of the levers are multiplicative, and the spread between combinations varies by orders of magnitude (in the pilot - by 80 times for K0).

In [ ]:
K5_BG = BG.K5_hours.mean()
SCR["K5_excess"] = (SCR.K5_hours - K5_BG) / N_CARS
SCR["import_volume"] = SCR.K2 / PREP["interface_defaults"]["L4-subsidy"]
# translate to English
print(f"background K5 without shock: {K5_BG:.0f} machine-hours per year ({K5_BG / N_CARS:.2f} h/machine)")

RESP = KPIS                                   # отклики для эффектов (K5 в лог-шкале - сам K5_hours)
for k in RESP:
    SCR["log_" + k] = np.log1p(SCR[k].clip(lower=0))

# средние по комбинациям (геометрические в лог-шкале - для графиков эффектов)
CELL = SCR.groupby(["shock", "design_id", "code"] + LEVERS)[KPIS + ["K5_excess", "import_volume", "price", "sec"]].mean().reset_index()
CELL.to_csv(OUT_DIR / "screening_cells.csv", index=False, encoding="utf-8-sig")
CELL.query("shock == 0.45").sort_values("K0").head(10).round(2)

## 1.6. Модель эффектов и ANOVA

Для каждого KPI и каждого уровня шока: `log(1+K) ~ (L2+L3+L4+L5+L6)**3 + C(seed)`, 160 прогонов. Взаимодействия 4-го и 5-го порядка уходят в остаток (стандартное допущение иерархии эффектов; при необходимости их можно добавить).

* **Эффект** члена = 2β в лог-шкале; `эффект, %` = 100·(e^{2β} − 1) - на сколько процентов меняется KPI при включении рычага, в среднем по остальным рычагам. Для взаимодействия - насколько эффект одного рычага меняется в зависимости от другого.
* **Частичная η²** = SS_член / (SS_член + SS_остаток) - доля разброса, объяснимая членом. Это мера важности; p-значения даются для справки (при 160 прогонах «значимо» почти всё).
* План ортогонален и сбалансирован по seed, поэтому оценки членов независимы, а тип сумм квадратов не влияет на результат.

In [ ]:
FORMULA = "y ~ (L2 + L3 + L4 + L5 + L6)**3 + C(seed)"
rows, MODELS = [], {}
for sh in SHOCKS:
    d = SCR[SCR.shock == sh]
    for k in RESP:
        y = d["log_" + k]
        if y.std() < 1e-12:                   
            continue
        m = smf.ols(FORMULA, data=d.assign(y=y)).fit()
        MODELS[(sh, k)] = m
        an = sm.stats.anova_lm(m, typ=2)
        ss_res = an.loc["Residual", "sum_sq"]
        for term in an.index:
            if term in ("Residual", "C(seed)"):
                continue
            beta = m.params[term]
            rows.append({"shock": sh, "KPI": k, "term": term.replace(":", "×"),
                         "order": term.count(":") + 1, "effect_log": 2 * beta,
                         "effect_%": 100 * (np.exp(2 * beta) - 1),
                         "ci95_%": 100 * (np.exp(2 * 1.96 * m.bse[term]) - 1),
                         "eta2_partial": an.loc[term, "sum_sq"] / (an.loc[term, "sum_sq"] + ss_res),
                         "p": an.loc[term, "PR(>F)"]})
EFFECTS = pd.DataFrame(rows)
EFFECTS.to_csv(OUT_DIR / "screening_effects.csv", index=False, encoding="utf-8-sig")
R2 = pd.DataFrame({f"{sh}": {k: MODELS[(sh, k)].rsquared for k in RESP if (sh, k) in MODELS} for sh in SHOCKS})
print("R² of models (proportion of variance explained by levers and seed):"); display(R2.round(3))
EFFECTS.query("shock == 0.45 and KPI == 'K0'").sort_values("eta2_partial", ascending=False).head(10).round(3)

## 1.7. Диаграммы Парето эффектов

Для каждого KPI - 12 самых важных членов по частичной η² при каждом шоке. Цвет столбца - знак эффекта (рост или снижение KPI; для всех KPI меньше = лучше). Пунктир - порог отбора 0.05.

In [ ]:
def pareto_effects(kpi, top=12):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)
    for ax, sh in zip(axes, SHOCKS):
        e = EFFECTS.query("KPI == @kpi and shock == @sh").nlargest(top, "eta2_partial").iloc[::-1]
        if e.empty:
            ax.set_title(f"shock {sh}: KPI = 0"); continue
        colors = [PAL[1] if v > 0 else PAL[0] for v in e["effect_log"]]
        ax.barh(e.term, e.eta2_partial, color=colors, height=0.7)
        for y, (eta, eff) in enumerate(zip(e.eta2_partial, e["effect_%"])):
            ax.text(eta + 0.01, y, f"{eff:+.0f}%", va="center", fontsize=8, color="#52514e")
        ax.axvline(ETA2_MIN, color="#52514e", lw=1, ls="--")
        ax.set_title(f"shock-depth = {sh}"); ax.set_xlim(0, 1.15)
        # translate to English
    axes[0].set_ylabel("levers and interactions"); axes[0].set_xlabel("partial η²  (blue - reduces KPI, orange - increases)")    
    axes[1].set_xlabel("partial η²  (blue - reduces KPI, orange - increases)")
    fig.suptitle(f"{kpi}: importance of levers and interactions", x=0.01, ha="left")
    plt.tight_layout(); plt.show()

for k in ["K0", "K0@90", "K1", "K4", "K5_hours", "K2", "K3_volume"]:
    pareto_effects(k)

## 1.8. Тепловая карта важности

Строки - главные эффекты и взаимодействия, у которых η² ≥ 0.05 хоть где-то; столбцы - KPI при каждом шоке. Один рисунок на весь скрининг (кандидат на слайд).

In [ ]:
piv = EFFECTS.pivot_table(index="term", columns=["KPI", "shock"], values="eta2_partial")
piv = piv.reindex(columns=pd.MultiIndex.from_product([[k for k in RESP if k in piv.columns.get_level_values(0)], SHOCKS]))
keep = piv.max(axis=1) >= ETA2_MIN
piv = piv[keep]
piv = piv.loc[sorted(piv.index, key=lambda t: (t.count("×"), t))]
fig, ax = plt.subplots(figsize=(1.0 + 0.42 * piv.shape[1], 0.6 + 0.3 * piv.shape[0]))
im = ax.imshow(piv.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(piv)), piv.index)
ax.set_xticks(range(piv.shape[1]), [f"{k}\n{s}" for k, s in piv.columns], fontsize=7)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.iat[i, j]
        if np.isfinite(v) and v >= ETA2_MIN:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if v > 0.6 else "#0b0b0b")
ax.grid(False); fig.colorbar(im, ax=ax, label="частичная η²", shrink=0.6)
ax.set_ylabel("levers and interactions"); ax.set_xlabel("KPI × shock-depth")
plt.tight_layout(); plt.show()

## 1.9. Графики взаимодействий

Геометрическое среднее KPI (по seed и остальным рычагам) при выключенном и включённом рычаге X, отдельные линии - второй рычаг Y. Параллельные линии - взаимодействия нет; расходящиеся или пересекающиеся - эффект X зависит от Y. По умолчанию рисуются 3 самых важных парных взаимодействия для K0 и K4 при среднем шоке; список можно задать вручную.

In [ ]:
def gmean_by(d, cols, kpi):
    return d.groupby(cols)["log_" + kpi].mean().pipe(np.expm1)

def interaction_plot(x, y, kpis=("K0", "K1", "K4", "K5_hours")):
    fig, axes = plt.subplots(len(kpis), 3, figsize=(12, 2.6 * len(kpis)), squeeze=False)
    for i, k in enumerate(kpis):
        for j, sh in enumerate(SHOCKS):
            ax = axes[i, j]
            g = gmean_by(SCR[SCR.shock == sh], [y, x], k)
            for c, lev in zip([PAL[0], PAL[1]], [-1, 1]):
                ax.plot([0, 1], [g.loc[(lev, -1)], g.loc[(lev, 1)]], "-o", color=c, ms=6,
                        label=f"{y} {'on' if lev == 1 else 'off'}")
            ax.set_xticks([0, 1], [f"{x} off", f"{x} on"]); ax.set_xlim(-0.2, 1.2)
            if i == 0: ax.set_title(f"shock-depth = {sh}")
            if j == 0: ax.set_ylabel(k)
    axes[0, 0].legend(frameon=False, fontsize=8)
    
    fig.suptitle(f"Interaction of {x} × {y}", x=0.01, ha="left"); plt.tight_layout(); plt.show()

top2 = (EFFECTS.query("order == 2 and shock == 0.45 and KPI in ['K0', 'K4']")
        .groupby("term").eta2_partial.max().nlargest(3).index.tolist())

print("most important pairwise interactions:", top2)
for t in top2:
    a, b = t.split("×")
    interaction_plot(a, b)

In [ ]:
# Triple interaction L6 x L2 x L4: price cap effect under different supply support
def triple_plot(kpis=("K0", "K4", "K5_hours")):
    fig, axes = plt.subplots(len(kpis), 3, figsize=(12, 2.6 * len(kpis)), squeeze=False)
    for i, k in enumerate(kpis):
        for j, sh in enumerate(SHOCKS):
            ax = axes[i, j]
            g = gmean_by(SCR[SCR.shock == sh], ["L2", "L4", "L6"], k)
            for c, (l2, l4) in zip(PAL, [(-1, -1), (1, -1), (-1, 1), (1, 1)]):
                lbl = {(-1, -1): "no L2 or L4", (1, -1): "L2 only", (-1, 1): "L4 only", (1, 1): "L2 and L4"}[(l2, l4)]
                ax.plot([0, 1], [g.loc[(l2, l4, -1)], g.loc[(l2, l4, 1)]], "-o", color=c, ms=6, label=lbl)
            ax.set_xticks([0, 1], ["L6 off", "L6 on"]); ax.set_xlim(-0.2, 1.2); ax.set_yscale("symlog", linthresh=1)
            if i == 0: ax.set_title(f"shock-depth = {sh}")
            if j == 0: ax.set_ylabel(f"{k} (log scale)")
    axes[0, 0].legend(frameon=False, fontsize=8)
    
    fig.suptitle("Price cap L6 under different supply support (L2, L4)", x=0.01, ha="left")
    plt.tight_layout(); plt.show()

triple_plot()

## 1.10. Проверка допущений модели

Если лог-преобразование оправдано, разброс остатков примерно одинаков во всех 32 ячейках (тест Левене: сравниваем сырую и лог-шкалу) и остатки не показывают структуры относительно прогноза.

In [ ]:
lev = []
for sh in SHOCKS:
    d = SCR[SCR.shock == sh]
    for k in ["K0", "K1", "K4", "K5_hours"]:
        groups_raw = [g[k].to_numpy() for _, g in d.groupby("code")]
        groups_log = [g["log_" + k].to_numpy() for _, g in d.groupby("code")]
        lev.append({"shock": sh, "KPI": k,
                    "Levene p, raw": stats.levene(*groups_raw).pvalue,
                    "Levene p, log": stats.levene(*groups_log).pvalue,
                    "max/min sd, raw": np.nanmax([g.std() for g in groups_raw]) / max(np.nanmin([g.std() for g in groups_raw]), 1e-9),
                    "max/min sd, log": np.nanmax([g.std() for g in groups_log]) / max(np.nanmin([g.std() for g in groups_log]), 1e-9)})
display(pd.DataFrame(lev).round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, sh in zip(axes, SHOCKS):
    m = MODELS.get((sh, "K0"))
    if m is not None:
        ax.scatter(m.fittedvalues, m.resid, s=8, color=PAL[0], alpha=0.6)
        ax.axhline(0, color="#52514e", lw=1)
    ax.set_title(f"K0, shock {sh}"); ax.set_xlabel("fitted log(1+K0)")
axes[0].set_ylabel("residual")
plt.tight_layout(); plt.show()

## 1.11. Отбор рычагов для этапа 2

In [ ]:
sel = []
for l in LEVERS:
    e = EFFECTS[EFFECTS.term.str.split("×").apply(lambda parts: l in parts)]
    best = e.loc[e.eta2_partial.idxmax()]
    main = EFFECTS.query("term == @l")
    sel.append({"lever": l, "max η² (any term)": best.eta2_partial,
                "where": f"{best.term}, {best.KPI}, shock {best.shock}",
                "max η² of main effect": main.eta2_partial.max(),
                "keep": best.eta2_partial >= ETA2_MIN})
SELECTION = pd.DataFrame(sel)
SELECTION.to_csv(OUT_DIR / "screening_selection.csv", index=False, encoding="utf-8-sig")
SELECTION.round(3)

## 1.12. Таблица 32 комбинаций и недоминируемые комбинации

KPI нормированы для чтения: K0, K1, K4 - доля от «none» при том же шоке (1 = как без мер); K5 - часы на машину сверх фона; K2 - m.u.; K3 - литры низкосортного бензина на литр дефицита «none» (масштаб кризиса). Комбинация **недоминируема** при данном шоке, если ни одна другая не лучше или равна ей по всем шести KPI и строго лучше хотя бы по одному. С шестью критериями недоминируемых будет много - это нормально для скрининга; цель - отбросить явно плохие.

In [ ]:
OBJ = ["K0", "K1", "K2", "K3_volume", "K4", "K5_hours"]

def nondominated(F):
    F = np.asarray(F, float)
    nd = np.ones(len(F), bool)
    for i in range(len(F)):
        dom = np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)
        nd[i] = not dom.any()
    return nd

tabs = []
for sh in SHOCKS:
    c = CELL[CELL.shock == sh].copy()
    none = c[c.design_id == "none"].iloc[0]
    c["nondominated"] = nondominated(c[OBJ])
    c["K0_rel"] = c.K0 / none.K0 if none.K0 > 0 else np.nan
    c["K1_rel"] = c.K1 / none.K1
    c["K4_rel"] = c.K4 / none.K4
    c["K5_h_per_car"] = c.K5_excess
    c["K3_per_L_deficit"] = c.K3_volume / none.K0 if none.K0 > 0 else np.nan
    tabs.append(c)
COMBO_TABLE = pd.concat(tabs)
nd_count = COMBO_TABLE.groupby("design_id").nondominated.sum()
COMBO_TABLE["nd_shocks"] = COMBO_TABLE.design_id.map(nd_count)
COMBO_TABLE.to_csv(OUT_DIR / "screening_combos.csv", index=False, encoding="utf-8-sig")

dropped = nd_count[nd_count == 0].index.tolist()
print(f"доминируемы при всех шоках ({len(dropped)}):", dropped)
cols = ["design_id", "K0_rel", "K1_rel", "K4_rel", "K5_h_per_car", "K2", "K3_per_L_deficit", "price", "nondominated", "nd_shocks"]
(COMBO_TABLE.query("shock == 0.45")[cols].sort_values(["nondominated", "K0_rel"], ascending=[False, True])
 .round(3).reset_index(drop=True))

## 1.13. Итог этапа

Сохраняется `screening_summary.json`: что оставить на этап 2, какие комбинации отброшены, время прогона (для бюджета этапов 2-3). Пришлите вывод этой ячейки, тепловую карту (1.8), графики L6 × L2 × L4 (1.9) и таблицу отбора (1.11).

In [ ]:
SUMMARY = {
    "created": time.strftime("%Y-%m-%d %H:%M"), "model_sha256": sha, "design": "full 2^5",
    "R": R, "seeds": SCR_SEEDS, "shocks": SHOCKS, "lever_on": LEVER_ON, "eta2_min": ETA2_MIN,
    "n_runs": int(len(SCR) + len(BG)), "sec_per_run_mean": float(SCR.sec.mean()),
    "n_workers": N_WORKERS, "runs_per_hour": RUNS_PER_HOUR, "k5_background": float(K5_BG),
    "levers_keep": SELECTION.loc[SELECTION["оставить"], "рычаг"].tolist(),
    "levers_drop": SELECTION.loc[~SELECTION["оставить"], "рычаг"].tolist(),
    "combos_dominated_everywhere": dropped,
    "top_terms": {f"{r.KPI} @ {r.shock}": r.term for r in EFFECTS.sort_values("eta2_partial", ascending=False)
                  .drop_duplicates(["KPI", "shock"]).itertuples()},
}
(OUT_DIR / "screening_summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=float), encoding="utf-8")
print(json.dumps({k: v for k, v in SUMMARY.items() if k != "top_terms"}, indent=1, ensure_ascii=False, default=float))